# Notebook 0 — Setup: environment, install, and data

Get the repository running so the rest of the workshop has something to build on. You will verify the Python environment, install the packages, and check that the input data is in place. Detailed setup is also in `../../src/README.md`.


## Learning objectives

- Confirm Python 3.10+ and the core scientific packages.
- Make the `model` and `web` packages importable.
- Download or locate the external datasets referenced by `src/model/config.yaml`.
- Verify the screening configuration loads.


## 0.1 Python version

The project requires Python 3.10 or newer. The intended environment is documented in `../AGENTS.md`; it is a dedicated conda environment with numpy, scipy, rasterio, netCDF4, xarray, numba, and matplotlib.


In [1]:
import platform
import sys

print('Python', sys.version.split()[0], '|', platform.platform())


Python 3.13.15 | macOS-26.5.2-arm64-arm-64bit-Mach-O


## 0.2 Install the packages

From the repository root:

```bash
python -m venv .venv && source .venv/bin/activate
pip install -e .
pip install -r src/requirements-lock.txt
# development extras (optional):
pip install -r src/requirements-dev.txt
```

An editable install (`pip install -e .`) puts `model` and `web` on the import path, so the notebooks below work without setting `PYTHONPATH`. If you prefer not to install, export `PYTHONPATH=src` before starting Jupyter.


In [2]:
try:
    from model.grid import StructuredGrid
    from model.solver import ShallowWaterSolver
    from model.forcing import make_synthetic_tidal_boundary
    from model.utils import speed, power_density
    from model.config import load_config
    from model.output import NetCDFStreamWriter
    print('model package: OK')
except ImportError as exc:
    print('model package not importable:', exc)
    print('Run:  pip install -e .   or   export PYTHONPATH=src')

for pkg in ('numpy', 'scipy', 'matplotlib', 'rasterio', 'netCDF4', 'xarray'):
    try:
        mod = __import__(pkg)
        print(f'{pkg:<12s}', getattr(mod, '__version__', '?'))
    except ImportError:
        print(f'{pkg:<12s} MISSING')


model package: OK
numpy        2.5.2
scipy        1.18.1
matplotlib   3.11.1
rasterio     1.5.1


netCDF4      1.7.4
xarray       2026.7.0


## 0.3 Download the data

The model can run on synthetic data with no files, but a realistic study needs external datasets:

| Dataset | Used for | How to get it |
|---------|----------|---------------|
| **GEBCO 2026** NetCDF | Bathymetry | `python downloader.py --gebco` (auto) |
| **GOT4.10c** harmonics | Tidal forcing | `python downloader.py --tidal` (auto) |
| **Philippines landmass GeoJSON** | Land mask | `python downloader.py --landmask` (auto) |
| FES2014 / TPXO9 | Alternative forcing | Manual (registration; see `../../src/README.md`) |

```bash
python downloader.py --all
```

`--all` auto-downloads the GEBCO subset, the GeoBoundaries landmask, and GOT4.10c, and prints the manual steps for FES2014 and TPXO9. Files already present are skipped.


## 0.4 Verify the configuration

`src/model/config.yaml` is the single source of truth for the screening run. Let's print the key sections and check the working directories exist.


In [3]:
import json
from pathlib import Path

try:
    from model.config import load_config
    cfg = load_config()
    print(json.dumps({
        'domain': cfg['domain'],
        'simulation': cfg['simulation'],
        'tidal_forcing': cfg['tidal_forcing'],
        'output': cfg['output'],
        'engine': cfg['engine'],
    }, indent=2, default=str))
except Exception as exc:
    print('config could not be loaded:', exc)

print('\n--- directories ---')
for name in ('data', 'output', 'cases'):
    print(('exists ' if Path(name).exists() else 'missing'), name)


{
  "domain": {
    "lon_min": 116.0,
    "lon_max": 128.0,
    "lat_min": 4.0,
    "lat_max": 22.0,
    "resolution_km": 2.0
  },
  "simulation": {
    "start_time": "2024-01-01T00:00:00",
    "duration_days": 15,
    "dt": null,
    "cfl_safety": 0.5,
    "cd": 0.0025,
    "ah": 0.0,
    "advection": false,
    "use_numba": null,
    "rho": 1025.0
  },
  "tidal_forcing": {
    "source": "got",
    "path": "data/GOT4.10c/grids_oceantide_netcdf/",
    "constituents": [
      "M2",
      "S2",
      "K1",
      "O1"
    ]
  },
  "output": {
    "dir": "output/",
    "save_interval_hours": 1,
    "results_nc": "results.nc",
    "final_geotiff": "tidal_power_density.tif",
    "max_speed_geotiff": "max_current_speed.tif",
    "bathymetry_geotiff": "bathymetry.tif",
    "distance_geotiff": "distance_to_coast.tif",
    "hotspots_geojson": "hotspots.geojson",
    "hotspot_threshold": 200.0
  },
  "engine": {
    "name": "python"
  }
}

--- directories ---
missing data
missing output
missing c

## 0.5 Conda quick tour (recommended)

The project is developed and tested in a dedicated **conda** environment named `tidaloss` (see `../AGENTS.md`). Conda bundles the compiled scientific stack — rasterio, netCDF4, numba — that plain `pip` wheels frequently struggle with, so it is the recommended way to reproduce this setup. Create and activate the environment, then follow § 0.2 for the pip installs:

```bash
conda create -n tidaloss python=3.12
conda activate tidaloss
# now run the pip install commands from § 0.2
```

Every shell command in this workshop runs inside the activated environment. Common conda commands:

| Command | Purpose |
|---------|---------|
| `conda create -n tidaloss python=3.12` | Create a new environment with a Python version |
| `conda activate tidaloss` / `conda deactivate` | Enter / leave the environment |
| `conda env list` | List all environments and their paths |
| `conda list` | Show packages installed in the active environment |
| `conda install numpy` | Install or update a package |
| `conda env export > environment.yml` | Snapshot the environment for sharing / backup |
| `conda env create -f environment.yml` | Recreate an environment from a snapshot |

The cell below only reports the conda version and environments; it does not create anything.


In [4]:
import shutil
import subprocess

if shutil.which('conda'):
    try:
        r = subprocess.run(['conda', '--version'],
                           capture_output=True, text=True, timeout=20)
        print((r.stdout or r.stderr).strip())
        r = subprocess.run(['conda', 'env', 'list'],
                           capture_output=True, text=True, timeout=20)
        print((r.stdout or r.stderr).strip())
    except Exception as exc:
        print('conda check failed:', exc)
else:
    print('conda not found on PATH (install Miniforge/Miniconda)')


conda not found on PATH (install Miniforge/Miniconda)


## 0.6 Docker quick tour (optional)

Docker is used for the two parts of the stack that are impractical to install natively:

- the **web service** (Flask + MapLibre map) ships as an image built from `src/Dockerfile` and run via `docker compose`;
- the **TELEMAC-2D refinement engine** runs *only* inside a pinned public Docker image (`flussplan/telemac:v8-latest` by default, see `src/model/config.yaml`), so the repo never compiles it.

An image is a frozen snapshot of a filesystem; a container is a running instance of it. `docker compose` turns `docker-compose.yml` into one-command workflows. The commands this repo actually uses:

| Command | Purpose |
|---------|---------|
| `docker compose up -d --build` | Build + start the web service at http://localhost:8001 |
| `docker compose ps` | Show container status |
| `docker compose logs -f tidal-web` | Follow the web service logs |
| `docker compose down` | Stop the containers (output data persists on disk) |
| `docker build -t tidal-model -f src/Dockerfile .` | Build the web image by hand |
| `docker run -p 5000:5000 -v "$(pwd)/output:/output" tidal-model` | Run the image directly (map port 5000, mount outputs) |
| `docker image ls` | List local images |
| `docker ps -a` | List running / stopped containers |

The cell below only reports availability; it does not start a container.


In [5]:
import shutil
import subprocess

if shutil.which('docker'):
    try:
        r = subprocess.run(['docker', 'compose', 'version'],
                           capture_output=True, text=True, timeout=20)
        print((r.stdout or r.stderr).strip())
    except Exception as exc:
        print('docker compose check failed:', exc)
else:
    print('docker not found on PATH (needed for the web + TELEMAC steps)')


Docker Compose version v5.1.3


## Next

Environment is ready. Move to [Notebook 1 — concept](1.concept.ipynb) for the physics behind the resource estimate.

---

[Index](README.md) · [← README.md](README.md) · [1.concept.ipynb →](1.concept.ipynb)
